In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from jax import jit
import matplotlib.pyplot as plt
import diffrax as dfx

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.parameters import BASE_PARAMS
from tb_macro.calibration import make_log_likelihood, get_runner
from tb_macro.plotting import plot_comp_distributions, plot_dynamic_mixing_matrix

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
runner, istate = get_runner(epi_model)

In [ ]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
calib_params = [
    "raw_transmission_rate",
    "detect_val_2", 
    "rel_sus_contained",
    "rel_sus_cleared",
    "rel_sus_children",
    "breakdown_rate",
    "clearance_rate",
    "clinical_progression_rate",
    "infectiousness_gain_rate",
]

param_bounds = {
    "raw_transmission_rate": [5.0, 18.0],
    "detect_val_2": [0.4, 0.7],
    "rel_sus_contained": [0.2, 0.5],
    "rel_sus_cleared": [0.5, 1.0],
    "rel_sus_children": [0.5, 1.0],
    "breakdown_rate": [0.01, 1.0],
    "clearance_rate": [0.01, 0.1],
    "clinical_progression_rate": [0.5, 5.0],
    "infectiousness_gain_rate": [0.5, 5.0],
}

def vector_to_params(calib_params, x):
    return dict(zip(calib_params, x))


def params_to_vector(calib_params, params):
    return np.array([params[p] for p in calib_params])


def params_to_bounds(calib_params, bounds):
    return [bounds[p] for p in calib_params]


In [ ]:
# Optimisation
log_like = make_log_likelihood(epi_model, disease_state, solver_kwargs, who_mort)

@jit
def opt_cr(x):
    return -log_like(BASE_PARAMS | vector_to_params(calib_params, x))

opt_res = minimize(opt_cr, params_to_vector(calib_params, BASE_PARAMS), method="Nelder-Mead", bounds=params_to_bounds(calib_params, param_bounds))
opt_res.x

In [ ]:
# Optimisation results
opt_param = vector_to_params(calib_params, opt_res.x)
results = epi_model.run(BASE_PARAMS | opt_param, solver_kwargs=solver_kwargs)

In [ ]:
results["compartments"].sumcats(compartment=clin_strat.categories()).to_pandas_df()["subclin"].clip(0.0)

In [ ]:
from importlib import reload

In [ ]:
from tb_macro import plotting

In [ ]:
reload(plotting)
plotting.plot_comp_distributions(results, disease_state, age_strat, infect_strat, clin_strat, END_TIME, group_popsize)

In [ ]:
plot_dynamic_mixing_matrix(results["computed_values"]["dynamic_mm"], 1970.0, 10.0, 3)

In [ ]:
from tb_macro.constants import LATENT_STATES
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET

In [ ]:
who_mort

In [ ]:
def plot_single_run_comparison():
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

    # Notifications
    notif_ax = axes[0, 0]
    notifs_modelled = results["flows"]["detection"].sum(to_dims="time").to_pandas_df()
    notifs_modelled.plot(ax=notif_ax, label="modelled")
    NOTIF_TARGET.plot(ax=notif_ax, linewidth=0.0, marker="o", label="target")
    notif_ax.set_xlim(2000, 2025)
    notif_ax.legend()
    notif_ax.set_title("notifications")

    # Latent
    latent_ax = axes[0, 1]
    total_pop = results["compartments"].sum(to_dims="time").to_pandas_df()
    latent_states = results["compartments"].query(compartment=disease_state[LATENT_STATES])
    latent_modelled = latent_states.sum(to_dims="time").to_pandas_df() / total_pop * 100.0
    latent_modelled.plot(ax=latent_ax, label="modelled")
    LATENT_TARGET.plot(ax=latent_ax, linewidth=0.0, marker="o", label="target")
    notif_ax.legend()
    latent_ax.set_title("latent")

    # Mortality
    mort_ax = axes[1, 0]
    community_death_age = results["flows"]["tb_mortality"].sum(to_dims="time").to_pandas_df()
    rx_death_age = results["flows"]["rx_death"].sum(to_dims="time").to_pandas_df()
    deaths = community_death_age + rx_death_age
    deaths.plot(ax=mort_ax)
    who_mort.plot(ax=mort_ax, linewidth=0.0, marker="o", label="target")
    mort_ax.set_title("mortality")
    axes[1, 1].set_axis_off()

    fig.tight_layout()
    plt.close()
    return fig

plot_single_run_comparison()